In [ ]:
!pip install transformers

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/drive/MyDrive/Trans-Ubiquitination-Colab/Ubi Human Equal Models

/content/drive/MyDrive/Trans-Ubiquitination-Colab/Ubi Human Equal Models


In [ ]:
import pandas as pd

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import torch
from transformers import BertTokenizer, BertModel
import logging

homo_sapiens = pd.read_excel("Ubi_Human_cdhit_Equal.xlsx", header = None, names = ["Sequence_with_len_21_positive", "Sequence_with_len_21_negative"])

print(homo_sapiens.head(3))

homo_sapiens_sd = homo_sapiens.iloc[:20000,0:2]

for i in range(0, len(homo_sapiens_sd)):
    homo_sapiens_sd.iloc[i][0] = homo_sapiens_sd.iloc[i][0].replace("", " ")
    homo_sapiens_sd.iloc[i][1] = homo_sapiens_sd.iloc[i][1].replace("", " ")

#print("homo_sapiens_sd:", homo_sapiens_sd)

print("length of homo_sapiens_sd:", len(homo_sapiens_sd))

#print(homo_sapiens_sd.iloc[0][0])

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

##################### Vector Creation for Positives ########################

n = "[CLS]" +  homo_sapiens_sd.iloc[0][0] + "[SEP]"

print(n)

positive_sentences = []
for i in range(0, len(homo_sapiens_sd)):
    positive_sentences.append("[CLS]" + homo_sapiens_sd.iloc[i][0] + "[SEP]")

#print("positive_sentences:", positive_sentences)

positive_sentences_tokenized = []
for i in range(0, len(positive_sentences)):
    tokenized_text = tokenizer.tokenize(str(positive_sentences[i]))
    positive_sentences_tokenized.append(tokenized_text)
positive_sentences_tokenized[0:4]

print("Total row number:", len(positive_sentences_tokenized))
print("Sequence length:", len(positive_sentences_tokenized[0]))

positive_sentences_indexes = []
for i in range(0, len(positive_sentences)):
    positive_sentences_indexes.append(tokenizer.convert_tokens_to_ids(positive_sentences_tokenized[i]))

positive_sentences_segment_ids = []

for i in range(0, int(len(positive_sentences)/2)):
    positive_sentences_segment_ids.append([1] * len(positive_sentences_tokenized[0]))
    positive_sentences_segment_ids.append([0] * len(positive_sentences_tokenized[0]))

positive_tokens_tensor = torch.tensor([positive_sentences_indexes])
#print(positive_tokens_tensor)

print("positive_tokens_tensor shape:", positive_tokens_tensor.shape)

positive_segments_tensors = torch.tensor([positive_sentences_segment_ids])
print(positive_segments_tensors)

print("positive_segments_tensors shape:", positive_segments_tensors.shape)

model_positive = BertModel.from_pretrained("bert-base-uncased", output_hidden_states = True)
model_positive.eval()

outputs_positive = []
with torch.no_grad():
    outputs_positive = model_positive(positive_tokens_tensor[0], positive_segments_tensors[0])
    positive_hidden_states = outputs_positive[2]

print ("Number of layers:", len(positive_hidden_states), "  (initial embeddings + 12 BERT layers)")
layer_i = 0

print ("Number of batches:", len(positive_hidden_states[layer_i]))
batch_i = 0

print ("Number of tokens:", len(positive_hidden_states[layer_i][batch_i]))
token_i = 0

print ("Number of hidden units:", len(positive_hidden_states[layer_i][batch_i][token_i]))

print("positive_hidden_states[0].shape:", positive_hidden_states[0].shape)


##################### Vector Creation for Negatives ########################

negative_sentences = []
for i in range(0, len(homo_sapiens_sd)):
    negative_sentences.append("[CLS]" + homo_sapiens_sd.iloc[i][1] + "[SEP]")

negative_sentences_tokenized = []
for i in range(0, len(negative_sentences)):
    tokenized_text = tokenizer.tokenize(str(negative_sentences[i]))
    negative_sentences_tokenized.append(tokenized_text)

print("Total row number:", len(negative_sentences_tokenized))
print("Sequence length:", len(negative_sentences_tokenized[0]))

negative_sentences_indexes = []
for i in range(0, len(negative_sentences)):
    negative_sentences_indexes.append(tokenizer.convert_tokens_to_ids(negative_sentences_tokenized[i]))

negative_sentences_segment_ids = []

for i in range(0, int(len(negative_sentences)/2)):
    negative_sentences_segment_ids.append([1] * len(negative_sentences_tokenized[0]))
    negative_sentences_segment_ids.append([0] * len(negative_sentences_tokenized[0]))

print("len(negative_sentences_segment_ids):", len(negative_sentences_segment_ids))

negative_tokens_tensor = torch.tensor([negative_sentences_indexes])
negative_tokens_tensor

negative_segments_tensors = torch.tensor([negative_sentences_segment_ids])

model_negative = BertModel.from_pretrained("bert-base-uncased", output_hidden_states = True)
model_negative.eval()

outputs_negative = []
with torch.no_grad():
    outputs_negative = model_negative(negative_tokens_tensor[0], negative_segments_tensors[0])
    negative_hidden_states = outputs_negative[2]

print("outputs_negative[0].shape:", outputs_negative[0].shape)

print ("Number of layers:", len(negative_hidden_states), "  (initial embeddings + 12 BERT layers)")
layer_i = 0

print ("Number of batches:", len(negative_hidden_states[layer_i]))
batch_i = 0

print ("Number of tokens:", len(negative_hidden_states[layer_i][batch_i]))
token_i = 0

print ("Number of hidden units:", len(negative_hidden_states[layer_i][batch_i][token_i]))

print ("CNN Preparation Part started!")

##################### CNN Preparation Part ########################

#### Positive part:

dataset_pos = np.array([])

for j in range(0, len(homo_sapiens_sd)):
    #print ("j1:", j)
    a = np.array([])
    for i in range(0,23):
        b = (np.array(positive_hidden_states[12][j][i]) + np.array(positive_hidden_states[11][j][i]) + np.array(positive_hidden_states[10][j][i]) + np.array(positive_hidden_states[9][j][i])) / 4
        a = np.hstack((a,b))
    if len(dataset_pos) == 0:
        dataset_pos = a
    else:
        dataset_pos = np.vstack((dataset_pos, a))
dataset_pos.shape
dataset_pos = pd.DataFrame(dataset_pos)


#### Negative part:

dataset_neg = np.array([])

for j in range(0, len(homo_sapiens_sd)):
    #print ("j2:", j)
    a = np.array([])
    for i in range(0,23):
        b = (np.array(negative_hidden_states[12][j][i]) + np.array(negative_hidden_states[11][j][i]) + np.array(negative_hidden_states[10][j][i]) + np.array(negative_hidden_states[9][j][i])) / 4
        a = np.hstack((a,b))
    if len(dataset_neg) == 0:
        dataset_neg = a
    else:
        dataset_neg = np.vstack((dataset_neg, a))
dataset_neg.shape
dataset_neg = pd.DataFrame(dataset_neg)

print("Dataset positive:", dataset_pos.shape)
print("Dataset negative:", dataset_neg.shape)

dataset_pos_labels = [1] * len(homo_sapiens_sd)
dataset_neg_labels = [0] * len(homo_sapiens_sd)

x = pd.concat([dataset_pos, dataset_neg], ignore_index = True)
y = dataset_pos_labels + dataset_neg_labels
print("Data type of x: ",type(x))
print("Data type of y: ", type(y))

print("x.shape", x.shape)

print("len(y):", len(y))

### Sending vector embeddings generated by BERT for the dataset with 208946 sample sizes:

directory_x = r"/content/drive/MyDrive/Trans-Ubiquitination-Colab/Ubi Human Equal Models/homo_ubi_embeddings20000_1_X.npy"
np.save(directory_x, x)
directoy_y = r"/content/drive/MyDrive/Trans-Ubiquitination-Colab/Ubi Human Equal Models/homo_ubi_embeddings20000_1_Y.npy"
np.save(directoy_y, y)


  Sequence_with_len_31_positive Sequence_with_len_31_negative
0         GGAFSGKDYTKVDRSAAYAAR         EGDFSVCRNCKRHVVSANFTL
1         GVNIPESHINKHLDSCLSREE         IKKEKEEAAKKRQEEQERKLK
2         NCQLSDPSSTKPGTLLKTSPS         RELHKLEGNLKLNRESMENLE
length of homo_sapiens_sd: 20000
[CLS] G G A F S G K D Y T K V D R S A A Y A A R [SEP]
Total row number: 20000
Sequence length: 23
positive_tokens_tensor shape: torch.Size([1, 20000, 23])
tensor([[[1, 1, 1,  ..., 1, 1, 1],
         [0, 0, 0,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 1, 1, 1],
         ...,
         [0, 0, 0,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 1, 1, 1],
         [0, 0, 0,  ..., 0, 0, 0]]])
positive_segments_tensors shape: torch.Size([1, 20000, 23])


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.bias', 'cls.seq_relationship.weight', 'cls.seq_relationship.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Number of layers: 13   (initial embeddings + 12 BERT layers)
Number of batches: 20000
Number of tokens: 23
Number of hidden units: 768
positive_hidden_states[0].shape: torch.Size([20000, 23, 768])
Total row number: 20000
Sequence length: 23
len(negative_sentences_segment_ids): 20000


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.bias', 'cls.seq_relationship.weight', 'cls.seq_relationship.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


outputs_negative[0].shape: torch.Size([20000, 23, 768])
Number of layers: 13   (initial embeddings + 12 BERT layers)
Number of batches: 20000
Number of tokens: 23
Number of hidden units: 768
CNN Preparation Part started!
Dataset positive: (20000, 17664)
Dataset negative: (20000, 17664)
Data type of x:  <class 'pandas.core.frame.DataFrame'>
Data type of y:  <class 'list'>
x.shape (40000, 17664)
len(y): 40000
